In [1]:
from utils import *
import pandas as pd
import os
import numpy as np

In [2]:
# load data 
file_path = os.path.join("tables", "exerciseTableForBN_python_processed.csv")
exerciseTable = pd.read_csv(file_path)

In [3]:
# 1. Define the Unrolled Time-Series Tiers
# We include empty lists for unused tiers so your constraint function works perfectly!
tiers_dbn = {
    'static': [], 
    'pre': ['startExerciseGlucoseLevel', 'IOBnormStartEx'], 
    'exercise': ['DurationValue', 'MET', 'CHODuringEx'],
    'outcome_during': [], 
    'end_exercise': ['endExerciseGlucoseLevel', 'IOBnormEndEx'],
    'outcome_post': ['postExerciseHypoEvent'] # Feel free to swap this with postExerciseTIR
}


all_features = exerciseTable.columns.tolist()
features_to_drop = set(all_features) - set(sum(tiers_dbn.values(), []))

In [4]:
# subset of sessions: aerobic activity with duration between 30 and 60 minutes

df = exerciseTable[(exerciseTable['ExerciseModality'] == 'Aerobic') & 
                (exerciseTable['DurationValue'] >= 20) & 
                (exerciseTable['DurationValue'] <= 60)].copy()

#df = exerciseTable.copy()

In [ ]:
df_discrete = discretize_data(df, 
    cv_strategy = "statistical",
    bmi_strategy = "clinical",
    hba1c_strategy = "clinical", 
    glucose_strategy = "clinical", 
    roc_strategy = "clinical",
    cols_to_remove = features_to_drop)

#Check Discretization
# check_discretization(df_discrete) # comment if not needed!


# Collapse bins
df_discrete_collapsed = collapse_sparse_bins(df_discrete)

In [ ]:
import pyagrum as gum
import pyagrum.lib.notebook as gnb



# 2. Create a list of just the columns we need for this specific network
# (Assuming 'df_aerobic' is your filtered 30-60 min aerobic dataframe)
dbn_columns = []
for tier_list in tiers_dbn.values():
    dbn_columns.extend(tier_list)

# Keep only these columns to force the AI to focus purely on the timeline
df_dbn_ready = df_discrete_collapsed[dbn_columns]

# 3. Initialize the Learner
learner_dbn = gum.BNLearner(df_dbn_ready)

# 4. Apply your masterpiece constraint function!
learner_dbn = apply_expert_constraints(learner_dbn, tiers_dbn)

# 5. Learn the Structure
learner_dbn.useLocalSearchWithTabuList() 
bn_dbn = learner_dbn.learnBN()

# 6. Visualize the Unrolled Time-Series Network
gnb.showBN(bn_dbn, size="14")